In [42]:
import duckdb
import polars as pl
from pathlib import Path
import plotly.express as px
import arrow

import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


from data_importers import LandRegistryImporter

Unfortunately the CSV file does not include a header row, so we specify these.

[How to access HM Land Registry Price Paid Data](https://www.gov.uk/guidance/about-the-price-paid-data) web site provides useful information to help us do this.  Here is the key information extracted from that site:

| Data item                         | Explanation (where appropriate)                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                    |
|-----------------------------------|----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------|
| Transaction unique identifier     | A reference number which is generated automatically recording each published sale. The number is unique and will change each time a sale is recorded.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               |
| Price                             | Sale price stated on the transfer deed.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               |
| Date of Transfer                  | Date when the sale was completed, as stated on the transfer deed.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                       |
| Postcode                          | This is the postcode used at the time of the original transaction. Note that postcodes can be reallocated and these changes are not reflected in the Price Paid Dataset.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         |
| Property Type                     | D = Detached, S = Semi-Detached, T = Terraced, F = Flats/Maisonettes, O = Other<br><br>**Note:**<br>- we only record the above categories to describe property type, we do not separately identify bungalows<br>- end-of-terrace properties are included in the Terraced category above<br>- ‘Other’ is only valid where the transaction relates to a property type that is not covered by existing values, for example where a property comprises more than one large parcel of land |
| Old/New                           | Indicates the age of the property and applies to all price paid transactions, residential and non-residential.<br><br>Y = a newly built property, N = an established residential building                                                                                                                                                                                                                                                                                                                                                                                                                                                                               |
| Duration                          | Relates to the tenure: F = Freehold, L = Leasehold etc.<br><br>Note that HM Land Registry does not record leases of 7 years or less in the Price Paid Dataset.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 |
| PAON                              | Primary Addressable Object Name. Typically the house number or name.                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 |
| SAON                              | Secondary Addressable Object Name. Where a property has been divided into separate units (for example, flats), the PAON (above) will identify the building and a SAON will be specified that identifies the separate unit/flat.                                                                                                                                                                                                                                                                                                                                                                                                                                             |
| Street                            |                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         |
| Locality                          |                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         |
| Town/City                         |                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         |
| District                          |                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         |
| County                            |                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         |
| PPD Category Type                 | Indicates the type of Price Paid transaction.<br><br>A = Standard Price Paid entry, includes single residential property sold for value.<br><br>B = Additional Price Paid entry including transfers under a power of sale/repossessions, buy-to-lets (where they can be identified by a Mortgage), transfers to non-private individuals and sales where the property type is classed as ‘Other’.<br><br>**Note:**<br>Note that category B does not separately identify the transaction types stated.<br>HM Land Registry has been collecting information on Category A transactions from January 1995. Category B transactions were identified from October 2013. |
| Record Status - monthly file only | Indicates additions, changes and deletions to the records (see guide below).<br><br>A = Addition<br>C = Change<br>D = Delete<br><br>**Note:**<br>Note that where a transaction changes category type due to misallocation (as above) it will be deleted from the original category type and added to the correct category with a new transaction unique identifier.                   |

There is a lot of interesting information in this dataset for us to analyse.  Let's first add column names in our query to make queries clear.

In [48]:
duckdb.sql(
    f"""
    SELECT
    column00 AS 'id',
    column01 AS 'price',
    column02 AS 'date',
    column03 AS 'postcode',
    column04 AS 'property_type',
    column05 AS 'old_new',
    column06 AS 'duration',
    column07 AS 'paon',
    column08 AS 'saon',
    column09 AS 'street',
    column10 AS 'locale',
    column11 AS 'town_city',
    column12 AS 'district',
    column13 AS 'county',
    column14 AS 'ppd_category',
    column15 AS 'record_type',
    year(date) AS 'year_of_sale',
    month(date) AS 'month_of_sale'
    FROM read_csv('{lri.DATA_FOLDER}/pp-*.csv');
    """
)

┌────────────────────────────────────────┬────────┬─────────────────────┬──────────┬───────────────┬─────────┬──────────┬───────────────────┬─────────┬──────────────────┬────────────────┬────────────────┬──────────────────────────────┬───────────────────────┬──────────────┬─────────────┬──────────────┬───────────────┐
│                   id                   │ price  │        date         │ postcode │ property_type │ old_new │ duration │       paon        │  saon   │      street      │     locale     │   town_city    │           district           │        county         │ ppd_category │ record_type │ year_of_sale │ month_of_sale │
│                varchar                 │ int64  │      timestamp      │ varchar  │    varchar    │ varchar │ varchar  │      varchar      │ varchar │     varchar      │    varchar     │    varchar     │           varchar            │        varchar        │   varchar    │   varchar   │    int64     │     int64     │
├───────────────────────────────────────

## Set up a database and load the raw data into a table

We're going to create a DuckDB database to persist the data we have extracted from the CSV files and then take it through a few transformation steps to finally generate some useful analytics over the data.

In [49]:
db = duckdb.connect(LAND_REGISTRY_DATABASE_PATH)

In [50]:
db.sql("SELECT * from information_schema.tables")

┌───────────────┬──────────────┬──────────────────────────────────────────┬────────────┬──────────────────────────────┬──────────────────────┬───────────────────────────┬──────────────────────────┬────────────────────────┬────────────────────┬──────────┬───────────────┬───────────────┐
│ table_catalog │ table_schema │                table_name                │ table_type │ self_referencing_column_name │ reference_generation │ user_defined_type_catalog │ user_defined_type_schema │ user_defined_type_name │ is_insertable_into │ is_typed │ commit_action │ TABLE_COMMENT │
│    varchar    │   varchar    │                 varchar                  │  varchar   │           varchar            │       varchar        │          varchar          │         varchar          │        varchar         │      varchar       │ varchar  │    varchar    │    varchar    │
├───────────────┼──────────────┼──────────────────────────────────────────┼────────────┼──────────────────────────────┼────────────────────

## Load raw data into a table

We have an empty database, let's start adding some tables!

First, we create a schema called "brzone" to allow us to organise the tables in the database according to a [medallion architecture](https://www.databricks.com/glossary/medallion-architecture).

Then we use a [CREATE TABLE](https://duckdb.org/docs/stable/sql/statements/create_table) to persist the data into table.

In [51]:
db.sql("CREATE SCHEMA IF NOT EXISTS bronze")

In [52]:
db.sql(
    f"""
    CREATE TABLE IF NOT EXISTS bronze.price_paid_raw AS
    SELECT
    column00 AS 'id',
    column01 AS 'price',
    column02 AS 'date',
    column03 AS 'postcode',
    column04 AS 'property_type',
    column05 AS 'old_new',
    column06 AS 'duration',
    column07 AS 'paon',
    column08 AS 'saon',
    column09 AS 'street',
    column10 AS 'locale',
    column11 AS 'town_city',
    column12 AS 'district',
    column13 AS 'county',
    column14 AS 'ppd_category',
    column15 AS 'record_type',
    FROM read_csv('{lri.DATA_FOLDER}/pp-*.csv');
    """
)

In [53]:
db.sql("SELECT * from information_schema.tables")

┌───────────────┬──────────────┬──────────────────────────────────────────┬────────────┬──────────────────────────────┬──────────────────────┬───────────────────────────┬──────────────────────────┬────────────────────────┬────────────────────┬──────────┬───────────────┬───────────────┐
│ table_catalog │ table_schema │                table_name                │ table_type │ self_referencing_column_name │ reference_generation │ user_defined_type_catalog │ user_defined_type_schema │ user_defined_type_name │ is_insertable_into │ is_typed │ commit_action │ TABLE_COMMENT │
│    varchar    │   varchar    │                 varchar                  │  varchar   │           varchar            │       varchar        │          varchar          │         varchar          │        varchar         │      varchar       │ varchar  │    varchar    │    varchar    │
├───────────────┼──────────────┼──────────────────────────────────────────┼────────────┼──────────────────────────────┼────────────────────

## Add features

Here we add some additional columns to provide useful features that will enable analysis of the data downstream:

- Extact the year and month from the date.  Using built in DuckDB [date part functions](https://duckdb.org/docs/stable/sql/functions/datepart).
- Use a [regular expression](https://en.wikipedia.org/wiki/Regular_expression) to extract the postcode area from the postcode - which is either the first one or first two letters from the postcode.
- Use a [CASE](https://duckdb.org/docs/stable/sql/expressions/case) statement to convert the single letter code for property type into a full description.


In [54]:
db.sql("CREATE SCHEMA IF NOT EXISTS silver")

In [55]:
db.sql("DROP VIEW IF EXISTS gold.price_paid_cleaned")

In [56]:
db.sql(
    f"""
    CREATE VIEW IF NOT EXISTS silver.price_paid_cleaned AS
    SELECT
    *,
    year(date) AS 'year_of_sale',
    month(date) AS 'month_of_sale',
    regexp_extract(postcode, '([A-Z]{{1,2}}).*', 1) AS 'postcode_area',
    CASE property_type
          WHEN 'D' THEN 'Detached'
          WHEN 'S' THEN 'Semi-Detached'
          WHEN 'T' THEN 'Terraced'
          WHEN 'F' THEN 'Flats/Maisonettes'
          WHEN 'O' THEN 'Other'
          ELSE property_type
    END AS 'property_type_full'
    FROM bronze.price_paid_raw;
    """
)

In [57]:
db.sql("SELECT * from information_schema.tables")

┌───────────────┬──────────────┬──────────────────────────────────────────┬────────────┬──────────────────────────────┬──────────────────────┬───────────────────────────┬──────────────────────────┬────────────────────────┬────────────────────┬──────────┬───────────────┬───────────────┐
│ table_catalog │ table_schema │                table_name                │ table_type │ self_referencing_column_name │ reference_generation │ user_defined_type_catalog │ user_defined_type_schema │ user_defined_type_name │ is_insertable_into │ is_typed │ commit_action │ TABLE_COMMENT │
│    varchar    │   varchar    │                 varchar                  │  varchar   │           varchar            │       varchar        │          varchar          │         varchar          │        varchar         │      varchar       │ varchar  │    varchar    │    varchar    │
├───────────────┼──────────────┼──────────────────────────────────────────┼────────────┼──────────────────────────────┼────────────────────

In [58]:
db.sql(
    """
    SELECT
    price,
    date,
    postcode,
    property_type,
    property_type_full,
    year_of_sale,
    month_of_sale,
    postcode_area    
    FROM silver.price_paid_cleaned LIMIT 10
    """
)

┌────────┬─────────────────────┬──────────┬───────────────┬────────────────────┬──────────────┬───────────────┬───────────────┐
│ price  │        date         │ postcode │ property_type │ property_type_full │ year_of_sale │ month_of_sale │ postcode_area │
│ int64  │      timestamp      │ varchar  │    varchar    │      varchar       │    int64     │     int64     │    varchar    │
├────────┼─────────────────────┼──────────┼───────────────┼────────────────────┼──────────────┼───────────────┼───────────────┤
│ 103750 │ 2016-09-21 00:00:00 │ CF62 6NT │ F             │ Flats/Maisonettes  │         2016 │             9 │ CF            │
│ 165000 │ 2016-09-16 00:00:00 │ NP10 9DN │ S             │ Semi-Detached      │         2016 │             9 │ NP            │
│  90000 │ 2016-09-02 00:00:00 │ SA11 3SD │ S             │ Semi-Detached      │         2016 │             9 │ SA            │
│ 105000 │ 2016-08-22 00:00:00 │ CF63 2QN │ T             │ Terraced           │         2016 │         

## Exploratory data analysis

Now we have the data loaded and tidied up, we can use duck db to analyse the data.  For example, let's see what the most expensive towns and cities are for domestic properties.

In the query below we:
- Select the `town_city` column because we want to aggregate the data to this level.
- Select the `property_type_full` column because we also want to differentiate between Detached, semi-detached, Terraced and Flats / Mainsonettes in our analysis.
- Average the `price` column casting the result to a decimal to make the results easier to read.
- Omit entries where the `property_type` is "O" which represents "Other", this category includes things like commercial properties, so we want to filter it out from the results in this case.
- Use the `GROUP BY ALL` which is a DuckDB enhancement which makes this type of query simpler.
- Sort the results in descending order.
- Take the top 10 results.

No surprise that all of the top results are for detached houses.

In [59]:
db.sql(
    f"""
    SELECT
    town_city,
    property_type_full,
    AVG(price)::DECIMAL(15, 2) AS average_price
    FROM silver.price_paid_cleaned
    WHERE property_type <> 'O'
    GROUP BY ALL
    ORDER BY average_price DESC
    LIMIT 10;
    """
)

┌──────────────────────┬────────────────────┬───────────────┐
│      town_city       │ property_type_full │ average_price │
│       varchar        │      varchar       │ decimal(15,2) │
├──────────────────────┼────────────────────┼───────────────┤
│ VIRGINIA WATER       │ Detached           │    2613071.45 │
│ LONDON               │ Detached           │    2114085.05 │
│ COBHAM               │ Detached           │    1707851.22 │
│ WEYBRIDGE            │ Detached           │    1698290.83 │
│ KINGSTON UPON THAMES │ Detached           │    1673400.46 │
│ TEDDINGTON           │ Detached           │    1556142.24 │
│ TWICKENHAM           │ Detached           │    1541824.40 │
│ BEACONSFIELD         │ Detached           │    1520775.18 │
│ ESHER                │ Detached           │    1478115.49 │
│ RADLETT              │ Detached           │    1474584.51 │
└──────────────────────┴────────────────────┴───────────────┘
  10 rows                                         3 columns

## Generating analytics

We are going to prepare some data for downstream consumption in a report.

This is an opportunity to show the creation of a VIEW.

In this case we want to generate a chart which shows details of average house price sales over time.



In [60]:
db.sql("CREATE SCHEMA IF NOT EXISTS gold")

In [61]:
db.sql("DROP VIEW IF EXISTS gold.average_price_by_month_and_property_type")

In [62]:
db.sql(
    f"""
    CREATE VIEW IF NOT EXISTS gold.average_price_by_month_and_property_type AS
    SELECT
    property_type_full,
    AVG(price) AS average_price,
    date_trunc('month', date) AS month
    FROM silver.price_paid_cleaned
    WHERE property_type <> 'O'
    GROUP BY ALL
    ORDER BY month, property_type_full;
    """
)

In [63]:
db.sql("FROM gold.average_price_by_month_and_property_type")

┌────────────────────┬────────────────────┬─────────────────────┐
│ property_type_full │   average_price    │        month        │
│      varchar       │       double       │      timestamp      │
├────────────────────┼────────────────────┼─────────────────────┤
│ Detached           │           150000.0 │ 1995-03-01 00:00:00 │
│ Detached           │            57500.0 │ 1995-04-01 00:00:00 │
│ Terraced           │            45000.0 │ 1995-04-01 00:00:00 │
│ Detached           │            85625.0 │ 1995-05-01 00:00:00 │
│ Semi-Detached      │           190000.0 │ 1995-05-01 00:00:00 │
│ Terraced           │            35000.0 │ 1995-05-01 00:00:00 │
│ Semi-Detached      │            46000.0 │ 1995-06-01 00:00:00 │
│ Detached           │            10000.0 │ 1995-09-01 00:00:00 │
│ Detached           │            65000.0 │ 1995-11-01 00:00:00 │
│ Detached           │            16000.0 │ 1996-01-01 00:00:00 │
│    ·               │               ·    │          ·          │
│    ·    

In [64]:
db.sql("SELECT * from information_schema.tables")

┌───────────────┬──────────────┬──────────────────────────────────────────┬────────────┬──────────────────────────────┬──────────────────────┬───────────────────────────┬──────────────────────────┬────────────────────────┬────────────────────┬──────────┬───────────────┬───────────────┐
│ table_catalog │ table_schema │                table_name                │ table_type │ self_referencing_column_name │ reference_generation │ user_defined_type_catalog │ user_defined_type_schema │ user_defined_type_name │ is_insertable_into │ is_typed │ commit_action │ TABLE_COMMENT │
│    varchar    │   varchar    │                 varchar                  │  varchar   │           varchar            │       varchar        │          varchar          │         varchar          │        varchar         │      varchar       │ varchar  │    varchar    │    varchar    │
├───────────────┼──────────────┼──────────────────────────────────────────┼────────────┼──────────────────────────────┼────────────────────

## Inspect the files

Let's compare the size of the raw CSV files with the database.

We can see that the database is less than half the size of the CSV files, thanks to the columnar storage applied by DuckDB.

This is impressive given we have persisted copies of the CSV data twice in both the Bronze and Silver schemas, and also extended the Silver table with more columns.

In [65]:
database_files = list(LAND_REGISTRY_DATABASE_PATH.parent.glob("*.db"))
csv_files = list(lri.DATA_FOLDER.glob("*.csv"))

In [66]:
files_to_analyse = pl.DataFrame(
    { 
        "file_name": [f.name for f in database_files + csv_files],
        "file_size": [f.stat().st_size for f in database_files + csv_files],
        "file_type": [f.suffix for f in database_files + csv_files]
    }
)

In [67]:
files_to_analyse

file_name,file_size,file_type
str,i64,str
"""pp-2021.csv""",223121026,""".csv"""
"""pp-2018.csv""",180162507,""".csv"""
"""pp-2017.csv""",185326793,""".csv"""
"""pp-2025.csv""",139672841,""".csv"""
"""pp-2023.csv""",149946179,""".csv"""
…,…,…
"""pp-2024.csv""",161058370,""".csv"""
"""pp-2016.csv""",181579035,""".csv"""
"""pp-2020.csv""",156180925,""".csv"""


In [68]:
files_to_analyse.group_by("file_type").agg(pl.col("file_size").sum().alias("total_file_size"))

file_type,total_file_size
str,i64
""".csv""",1766572499


In [69]:
# Calculate total sizes for .db and .csv files
db_size = files_to_analyse.filter(pl.col("file_type") == ".db")["file_size"].sum()
csv_size = files_to_analyse.filter(pl.col("file_type") == ".csv")["file_size"].sum()

# Calculate compression percentage
compression_percentage = (db_size / csv_size) * 100
print(f"DuckDB database is {compression_percentage:.1f}% of the original size of the CSV files.")

DuckDB database is 0.0% of the original size of the CSV files.


## Add visualisation over the analytics

Finally we convert the Gold `average_price_by_month_and_property_type` view to a Polars dataframe so we can visualise the data on a line chart using the Plotly Express charting package.

In [70]:
df = db.sql("FROM gold.average_price_by_month_and_property_type").pl()

In [71]:
df

property_type_full,average_price,month
str,f64,datetime[μs]
"""Detached""",150000.0,1995-03-01 00:00:00
"""Detached""",57500.0,1995-04-01 00:00:00
"""Terraced""",45000.0,1995-04-01 00:00:00
"""Detached""",85625.0,1995-05-01 00:00:00
"""Semi-Detached""",190000.0,1995-05-01 00:00:00
…,…,…
"""Terraced""",282592.043352,2026-01-01 00:00:00
"""Detached""",475917.920667,2026-02-01 00:00:00
"""Flats/Maisonettes""",262365.555645,2026-02-01 00:00:00


In [72]:
fig = px.line(
    df,
    x="month",
    y="average_price",
    color="property_type_full",
    line_dash="property_type_full",
    color_discrete_sequence=["#8CC600", "black", "darkgrey", "#EE7423"],
    line_dash_sequence=["solid", "dash", "dot", "dashdot"],
    title="Average Price by Month and Property Type"
)

fig.update_layout(
    xaxis_title="Month",
    yaxis_title="Average Price",
    legend_title="Property Type",
)

fig.show()

# Process monthly updates

Here the monthly updates utilises the "record_type" column where:

- a = add as a new record
- c = change / update existing record
- d = delete


## Latest Import Monthly Updates

In [73]:
lri.import_monthly_update()

INFO:data_importers.land_registry_importer:Downloading file pp-monthly-update-new-version.csv from http://prod.publicdata.landregistry.gov.uk.s3-website-eu-west-1.amazonaws.com/pp-monthly-update-new-version.csv
INFO:data_importers.land_registry_importer:File pp-monthly-update-new-version.csv downloaded successfully


In [74]:
db.sql(
    f"""
    DROP TABLE IF EXISTS bronze.monthly_updates;
    CREATE TABLE IF NOT EXISTS bronze.monthly_updates AS
    SELECT
    column00 AS 'id',
    column01 AS 'price',
    column02 AS 'date',
    column03 AS 'postcode',
    column04 AS 'property_type',
    column05 AS 'old_new',
    column06 AS 'duration',
    column07 AS 'paon',
    column08 AS 'saon',
    column09 AS 'street',
    column10 AS 'locale',
    column11 AS 'town_city',
    column12 AS 'district',
    column13 AS 'county',
    column14 AS 'ppd_category',
    column15 AS 'record_type',
    FROM read_csv('{lri.DATA_FOLDER}/{lri.MONTHLY_UPDATE_FILE_NAME}');
    """
)

In [75]:
db.sql(
    f"""
    SELECT record_type, COUNT(*)
    FROM bronze.monthly_updates
    GROUP BY record_type;
    """
)

┌─────────────┬──────────────┐
│ record_type │ count_star() │
│   varchar   │    int64     │
├─────────────┼──────────────┤
│ A           │        89083 │
│ C           │         2962 │
│ D           │         1452 │
└─────────────┴──────────────┘

## Apply Monthly Updates to Bronze

### Baseline Row Count

In [76]:
db.sql(
    f"""
    SELECT COUNT(*)
    FROM bronze.price_paid_raw;
    """
)

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     10146456 │
└──────────────┘

### Insert New "A" Records

In [77]:
db.sql(
    f"""
    INSERT INTO bronze.price_paid_raw
    SELECT * FROM bronze.monthly_updates
    WHERE record_type = 'A';
    """
)

In [78]:
db.sql(
    f"""
    SELECT COUNT(*)
    FROM bronze.price_paid_raw;
    """
)

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     10235539 │
└──────────────┘

### Delete "D" Records

In [79]:
db.sql(
    f"""
    DELETE FROM bronze.price_paid_raw
    WHERE id IN (SELECT id FROM bronze.monthly_updates WHERE record_type = 'D');
    """
)

In [80]:
db.sql(
    f"""
    SELECT COUNT(*)
    FROM bronze.price_paid_raw;
    """
)

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     10235539 │
└──────────────┘

### Update "C" Records

In [81]:
db.sql(
    f"""
    UPDATE bronze.price_paid_raw
    SET
        price = mu.price,
        date = mu.date,
        postcode = mu.postcode,
        property_type = mu.property_type,
        old_new = mu.old_new,
        duration = mu.duration,
        paon = mu.paon,
        saon = mu.saon,
        street = mu.street,
        locale = mu.locale,
        town_city = mu.town_city,
        district = mu.district,
        county = mu.county,
        ppd_category = mu.ppd_category,
        record_type = mu.record_type
    FROM bronze.monthly_updates AS mu
    WHERE bronze.price_paid_raw.id = mu.id AND mu.record_type = 'C';
    """
)

In [82]:
db.sql(
    f"""
    SELECT COUNT(*)
    FROM bronze.price_paid_raw;
    """
)

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│     10235539 │
└──────────────┘